In [ ]:
import pandas as pd
import re
import spacy
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Carregar os dois arquivos
df_train = pd.read_csv("../src/data/processed/df_train_processed.csv", index_col=0)

In [ ]:
df_train = df_train.head(5000)

In [ ]:
# Criar tabela de contagem por categoria
tabela_categorias = df_train["condition_label"].value_counts().reset_index()
tabela_categorias.columns = ["Categoria", "Quantidade"]

print(tabela_categorias)


In [ ]:
# modelo de baseline TF_IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.1, sublinear_tf=True)
tfidf = tv.fit_transform(df_train.medical_abstract_clean)
tfidf_df = pd.DataFrame(tfidf.toarray(), columns=tv.get_feature_names_out())
tfidf_df

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import pandas as pd

# 1. Vetorização TF-IDF
tv = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.1, sublinear_tf=True)
X = tv.fit_transform(df_train['medical_abstract_clean'])

# 2. Labels
y = df_train['condition_name']

# 3. Separar treino e teste do SMOTE (evita data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 4. SMOTE no treino
smote = SMOTE(random_state=42, k_neighbors=1)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# 5. Treinar modelo
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=50,
    random_state=42
)   
rf_model.fit(X_resampled, y_resampled)

# 6. Predições 
y_pred = rf_model.predict(X_test)

# 7. Avaliação
print(classification_report(y_test, y_pred))

In [ ]:
# Função para normalizar texto
def lower_replace(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)       # remove conteúdo entre colchetes
    text = re.sub(r'[^\w\s]', '', text)       # remove pontuação
    return text

# Tokenização + lematização + remoção de stopwords
def token_lemma_stop(text: str) -> list:
    doc = nlp(text)
    return [token.lemma_ for token in doc if not token.is_stop]

# Filtrar apenas certas classes gramaticais (exemplo: substantivos e adjetivos)
def filter_pos(tokens: list) -> str:
    doc = nlp(" ".join(tokens))
    return " ".join([token.text for token in doc if token.pos_ in ["NOUN", "ADJ", "PRON", "VERB"]])


# Pipeline único
def preprocess(text: str) -> list:
    text = lower_replace(text)
    tokens = token_lemma_stop(text)
    return filter_pos(tokens)


In [ ]:
# Exemplo de novo texto
novo_texto = "Sexually transmitted diseases of the colon, rectum, and anus. The challenge of the nineties. During the past two decades, an explosive growth in both the prevalence and types of sexually transmitted diseases has occurred. Up to 55 percent of homosexual men with anorectal complaints have gonorrhea"
# Pré-processar
novo_texto_proc = preprocess(novo_texto)
print(novo_texto_proc)

# Vetorizar com o TF-IDF já treinado
X_novo = tv.transform([novo_texto_proc])
df = pd.DataFrame(X_novo.toarray(), columns=tv.get_feature_names_out())

display(df)

# Predição
pred = rf_model.predict(X_novo)
print("Classe prevista:", pred[0])


In [ ]:
probs = rf_model.predict_proba(X_novo)
print("Probabilidades por classe:", probs)
